# Face Recognition — Base d'embeddings InsightFace (LFW Top-20)

**Projet** : Système de Reconnaissance Faciale — PFE Sonatel Academy  
**Auteur** : Ibrahima Gabar Diop  

Pipeline : RetinaFace (détection) + ArcFace ResNet50 (embeddings 512-dim)


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'insightface', 'onnxruntime-gpu', '-q'], check=True)


In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from insightface.app import FaceAnalysis

OUTPUT_DIR = Path('/kaggle/working')
REFS_DIR   = OUTPUT_DIR / 'face_refs'
REFS_DIR.mkdir(parents=True, exist_ok=True)

N_CLASSES         = 20
MIN_IMAGES        = 30
N_REFS_PER_PERSON = 5
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Exploration automatique des chemins
base = Path('/kaggle/input/lfw-dataset')
print('Contenu racine :')
for p in sorted(base.iterdir()):
    print(f'  {p}')

csv_candidates = list(base.rglob('*allnames*.csv')) + list(base.rglob('*_allnames*.csv'))
CSV_ALLNAMES = csv_candidates[0]
print(f'CSV : {CSV_ALLNAMES}')

lfw_deep = list(base.rglob('lfw-deepfunneled'))
LFW_DIR  = lfw_deep[0] if lfw_deep else list(base.rglob('lfw'))[0]
# Si le dossier contient un sous-dossier du meme nom, descendre d'un niveau
sub = LFW_DIR / LFW_DIR.name
if sub.exists():
    LFW_DIR = sub
print(f'Images : {LFW_DIR}')
print(f'Exemple : {next(LFW_DIR.rglob("*.jpg"), "aucune")}')


## 1. Sélection des classes


In [ ]:
df_all = pd.read_csv(CSV_ALLNAMES)
print(df_all.head())
top20  = df_all[df_all['images'] >= MIN_IMAGES].sort_values('images', ascending=False).head(N_CLASSES)
print(top20[['name','images']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top20['name'][::-1], top20['images'][::-1], color='steelblue')
ax.set_xlabel("Nombre d'images")
ax.set_title(f'LFW Top-{N_CLASSES}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_balance.png', dpi=150)
plt.show()


## 2. Chargement InsightFace


In [ ]:
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace buffalo_l charge (RetinaFace + ArcFace ResNet50)')


## 3. Generation des embeddings


In [ ]:
embeddings_db = {}
failed = []

for _, row in top20.iterrows():
    person = row['name']
    person_dir = LFW_DIR / person
    imgs = sorted(person_dir.glob('*.jpg'))
    if not imgs:
        failed.append(person)
        print(f'  SKIP  {person} (dossier introuvable : {person_dir})')
        continue
    random.shuffle(imgs)
    sample = imgs[:N_REFS_PER_PERSON]

    embs = []
    best_img_path = None
    best_area = 0

    for img_path in sample:
        img   = cv2.imread(str(img_path))
        faces = face_app.get(img)
        if not faces:
            continue
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        embs.append(face.embedding)
        area = (face.bbox[2]-face.bbox[0])*(face.bbox[3]-face.bbox[1])
        if area > best_area:
            best_area = area
            best_img_path = img_path

    if not embs:
        failed.append(person)
        print(f'  FAIL  {person} (aucun visage detecte)')
        continue

    mean_emb = np.mean(embs, axis=0)
    mean_emb = mean_emb / np.linalg.norm(mean_emb)
    embeddings_db[person] = mean_emb
    shutil.copy(best_img_path, REFS_DIR / f'{person}.jpg')
    print(f'  OK    {person:35s} {len(embs)} embs')

print(f'\nTotal : {len(embeddings_db)} OK, {len(failed)} echecs')


## 4. Export


In [ ]:
npz_path = OUTPUT_DIR / 'face_embeddings.npz'
np.savez(str(npz_path), **embeddings_db)
print(f'Embeddings : {npz_path} ({npz_path.stat().st_size/1024:.0f} KB)')
print(f'Refs       : {len(list(REFS_DIR.glob("*.jpg")))} images dans {REFS_DIR}')

loaded = np.load(str(npz_path))
print(f'Check : {list(loaded.keys())[:3]} ...')
print(f'Dim   : {loaded[list(loaded.keys())[0]].shape}')


In [ ]:
ref_imgs = sorted(REFS_DIR.glob('*.jpg'))
cols = 5
rows = max(1, (len(ref_imgs) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
axes = axes.flatten()
for i, img_path in enumerate(ref_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(img_path.stem[:18], fontsize=8)
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Images de reference exportees', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reference_faces.png', dpi=150)
plt.show()


In [ ]:
print('=' * 55)
print('  RESUME FINAL')
print('=' * 55)
print(f'  Modele     : InsightFace buffalo_l')
print(f'  Detecteur  : RetinaFace (det_10g.onnx)')
print(f'  Embeddings : ArcFace ResNet50 (w600k_r50.onnx) 512-dim')
print(f'  Identites  : {len(embeddings_db)}')
print(f'  A telecharger :')
print(f'    face_refs/*.jpg     -> data/faces/ du repo')
print(f'    face_embeddings.npz -> data/ (optionnel)')
print('=' * 55)
